# RAG — Retrieval Augmented Generation (Part 1)

### Build a complete RAG pipeline from scratch

---

This notebook is the **hands-on companion** to **Section 4 (Part 1)** on the course page.

We'll build a RAG system step by step over TechStore's internal documents:
- `return_policy.txt` — customer-facing return policy
- `employee_handbook.txt` — HR policies (PTO, remote work, benefits)
- `product_catalog.txt` — product descriptions and specs

**By the end:** You'll have a working Q&A system that answers questions from these documents accurately — no hallucination, no guessing.

**Tools used:** OpenAI (embeddings + generation), ChromaDB (vector database), LangChain (text splitting)

---

## Setup

In [ ]:
!pip install openai chromadb python-dotenv tiktoken langchain -q

In [27]:
!pip list |grep -E 'openai|chromadb|langchain'

chromadb                                 1.5.7
langchain                                0.3.28
langchain-core                           0.3.84
langchain-text-splitters                 0.3.11
openai                                   2.28.0
You should consider upgrading via the '/Volumes/work/teach/genai-beginner/.venv/bin/python3 -m pip install --upgrade pip' command.


In [14]:
!pip install langchain

     |████████████████████████████████| 1.0 MB 1.5 MB/s eta 0:00:01
     |████████████████████████████████| 396 kB 6.4 MB/s eta 0:00:01
     |████████████████████████████████| 2.2 MB 5.2 MB/s eta 0:00:01
     |████████████████████████████████| 459 kB 7.4 MB/s eta 0:00:01
     |████████████████████████████████| 604 kB 12.3 MB/s eta 0:00:01
     |████████████████████████████████| 54 kB 9.8 MB/s  eta 0:00:01
     |████████████████████████████████| 640 kB 9.3 MB/s eta 0:00:01
  Attempting uninstall: async-timeout
    Found existing installation: async-timeout 5.0.1
    Uninstalling async-timeout-5.0.1:
      Successfully uninstalled async-timeout-5.0.1
You should consider upgrading via the '/Volumes/work/teach/genai-beginner/.venv/bin/python3 -m pip install --upgrade pip' command.


In [13]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [15]:
import os
import json
import glob
import textwrap
import numpy as np
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY first.")

client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

def ask(prompt, system_prompt=None, temperature=0.3, model=LLM_MODEL, max_tokens=1000):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens
    )
    text = response.choices[0].message.content.strip()
    usage = response.usage
    print(text)
    print(f"\n{'─' * 50}")
    print(f"Tokens: {usage.total_tokens} | Cost: ~${usage.prompt_tokens * 0.15e-6 + usage.completion_tokens * 0.60e-6:.6f}")
    return text

def get_embeddings(texts, model=EMBEDDING_MODEL):
    response = client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

GROQ_API_KEY=os.environ.get("GROQ_API_KEY")
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"  # ← OpenAI-compatible endpoint
)

def ask_small(prompt, system_prompt=None, temperature=0.3, model="llama-3.1-8b-instant", max_tokens=1000):
    """Send a prompt to a SMALL model (8B params) to show why prompt engineering matters."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = groq_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    result = (response.choices[0].message.content or "").strip()
    usage = response.usage
    total = usage.total_tokens if usage else 0

    print(result)
    print(f"\n{'─' * 50}")
    print(f"Model: {model} (~8B params) | Tokens: {total} (in {usage.prompt_tokens}, out {usage.completion_tokens})")
    print(f"💡 Compare this output with ask() using GPT-4o-mini to see why prompting matters.")

    return result

print("✅ Setup complete.")

✅ Setup complete.


---

# Chapter 1: The Problem — Why We Need RAG

---

Let's first see the problem. Ask the LLM about TechStore — it doesn't know anything about this company.

In [ ]:
print("Question 1: What is TechStore's return policy for electronics?")
print("═" * 60)
ask("What is TechStore's return policy for electronics?")
print("\n\n small model:")
ask_small("What is TechStore's return policy for electronics?")

print("\n\n")
print("Question 2: How many PTO days does a TechStore employee with 4 years tenure get?")
print("═" * 60)
ask("How many PTO days does a TechStore employee with 4 years tenure get?")
print("\n\n small model:")
ask_small("How many PTO days does a TechStore employee with 4 years tenure get?")

print("\n\n💡 The model either hallucinated or said it doesn't know.")
print("   By the end of this notebook, it will answer both correctly.")

---

# Chapter 4: Reading & Parsing Documents

---

Step 1: Read the raw text from files into Python.

In [16]:
DOCS_DIR = "sample_docs"

documents = []
for filepath in sorted(glob.glob(f"{DOCS_DIR}/*.txt")):
    with open(filepath, "r") as f:
        content = f.read()
    doc_name = os.path.basename(filepath)
    documents.append({"name": doc_name, "content": content})
    print(f"📄 {doc_name} ({len(content):,} chars, ~{len(content)//4} tokens)")

print(f"\n✅ Read {len(documents)} documents")

📄 employee_handbook.txt (3,780 chars, ~945 tokens)
📄 product_catalog.txt (2,552 chars, ~638 tokens)
📄 return_policy.txt (3,567 chars, ~891 tokens)

✅ Read 3 documents


In [21]:
# Quick look at each document
for doc in documents:
    print(f"\n{'═' * 60}")
    print(f"📄 {doc['name']}")
    print(f"{'═' * 60}")
    print(doc['content'][:300] + "...")


════════════════════════════════════════════════════════════
📄 employee_handbook.txt
════════════════════════════════════════════════════════════
TechStore Employee Handbook — HR Policies
Version 3.2 | Effective March 2025

Chapter 1: Paid Time Off (PTO)

Full-time employees accrue PTO at the following rates based on years of service:
- 0-2 years: 15 days per year (1.25 days per month)
- 3-5 years: 20 days per year (1.67 days per month)
- 6+ ...

════════════════════════════════════════════════════════════
📄 product_catalog.txt
════════════════════════════════════════════════════════════
TechStore Product Catalog — Q1 2025

Category: Laptops

ProMax 15 — $1,299
Our flagship laptop. 15.6-inch 4K OLED display, Intel i9-14900H processor, 32GB RAM, 1TB NVMe SSD. Battery life up to 12 hours. Weight: 4.2 lbs. Includes 1-year premium warranty. Best for: professional users, video editing, s...

════════════════════════════════════════════════════════════
📄 return_policy.txt
══════════════════

---

# Chapter 5: Chunking

---

Documents are too long to embed whole. We split into chunks so retrieval is granular — find the specific paragraph, not the whole document.

**The way you chunk directly affects retrieval quality.** Bad chunking = wrong results = bad answers.

### The naive approach: Fixed-size chunking (and why it's bad)

In [18]:
def chunk_fixed(text, chunk_size=500, overlap=50):
    """Naive: split every N characters. Cuts mid-sentence."""
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


naive_chunks = chunk_fixed(documents[2]['content'])

print(f"Fixed-size → {len(naive_chunks)} chunks\n")
for i, chunk in enumerate(naive_chunks[:3]):
    print(f"Chunk {i+1} ends with: \"...{chunk[-50:]}\"")
    ends_clean = chunk.rstrip().endswith(('.', '!', '?', ':'))
    print(f"  Clean end? {'✅' if ends_clean else '❌ CUT MID-SENTENCE'}\n")

print("⚠️ Fixed-size cuts mid-sentence. The LLM gets incomplete facts.")
print("   Production systems use smarter chunking — next cell.")

Fixed-size → 8 chunks

Chunk 1 ends with: "...blets, phones, monitors, peripherals) have a short"
  Clean end? ❌ CUT MID-SENTENCE

Chunk 2 ends with: "...re products are non-refundable once the activation"
  Clean end? ❌ CUT MID-SENTENCE

Chunk 3 ends with: "... Lost or stolen gift cards cannot be replaced with"
  Clean end? ❌ CUT MID-SENTENCE

⚠️ Fixed-size cuts mid-sentence. The LLM gets incomplete facts.
   Production systems use smarter chunking — next cell.


### The production approach: LangChain RecursiveCharacterTextSplitter

This is what real RAG systems use. It tries to split by:
1. `\n\n` (paragraphs) — best
2. `\n` (lines) — if paragraphs are too big
3. `. ` (sentences) — if lines are too big
4. ` ` (words) — last resort

Chunks always end at natural boundaries.

In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

smart_chunks = splitter.split_text(documents[2]['content'])

print(f"Fixed-size → {len(naive_chunks)} chunks (mid-sentence cuts)")
print(f"LangChain  → {len(smart_chunks)} chunks (clean boundaries)\n")

for i, chunk in enumerate(smart_chunks[:5]):
    print(f"Chunk {i+1} ({len(chunk)} chars):")
    print(f"starts with: \"{chunk[:60]}...\"")
    print("--------------------")
    print(f"  Ends with: \"...{chunk[-60:]}\"")
    ends_clean = chunk.rstrip().endswith(('.', '!', '?', ':', ')'))
    print(f"  Clean? {'✅' if ends_clean else '⚠️'}\n")

print("✅ Clean sentence boundaries. 3 lines of code. Production-ready.")

Fixed-size → 8 chunks (mid-sentence cuts)
LangChain  → 9 chunks (clean boundaries)

Chunk 1 (391 chars):
starts with: "TechStore Return Policy
Last Updated: January 2025

1. Stand..."
--------------------
  Ends with: "...subject to a 15% restocking fee at the manager's discretion."
  Clean? ✅

Chunk 2 (466 chars):
starts with: "2. Electronics Return Policy
All electronics (laptops, table..."
--------------------
  Ends with: "...n under this policy — see the Damaged Items section instead."
  Clean? ✅

Chunk 3 (398 chars):
starts with: "3. Software and Digital Products
Software products are non-r..."
--------------------
  Ends with: "...und if less than 20% of the subscription period has elapsed."
  Clean? ✅

Chunk 4 (327 chars):
starts with: "4. Gift Cards
Gift cards are non-refundable and non-exchange..."
--------------------
  Ends with: "... a sales event may have different terms printed on the card."
  Clean? ✅

Chunk 5 (415 chars):
starts with: "5. Holiday Return Policy
Purchase

### Chunk all documents

In [22]:
all_chunks = []

for doc in documents:
    chunks = splitter.split_text(doc["content"])
    for i, chunk_text in enumerate(chunks):
        all_chunks.append({
            "id": f"{doc['name']}_chunk_{i}",
            "text": chunk_text,
            "source": doc["name"],
            "chunk_index": i,
        })
    print(f"📄 {doc['name']}: {len(chunks)} chunks")

print(f"\n✅ Total: {len(all_chunks)} chunks")
print(f"Average chunk size: {sum(len(c['text']) for c in all_chunks) // len(all_chunks)} chars")

📄 employee_handbook.txt: 15 chunks
📄 product_catalog.txt: 7 chunks
📄 return_policy.txt: 9 chunks

✅ Total: 31 chunks
Average chunk size: 318 chars


In [23]:
# Inspect a few chunks
for chunk in all_chunks[:4]:
    print(f"\n{'─' * 50}")
    print(f"ID: {chunk['id']} | Source: {chunk['source']}")
    print(textwrap.fill(chunk['text'], width=80))


──────────────────────────────────────────────────
ID: employee_handbook.txt_chunk_0 | Source: employee_handbook.txt
TechStore Employee Handbook — HR Policies Version 3.2 | Effective March 2025
Chapter 1: Paid Time Off (PTO)  Full-time employees accrue PTO at the following
rates based on years of service: - 0-2 years: 15 days per year (1.25 days per
month) - 3-5 years: 20 days per year (1.67 days per month) - 6+ years: 25 days
per year (2.08 days per month)

──────────────────────────────────────────────────
ID: employee_handbook.txt_chunk_1 | Source: employee_handbook.txt
PTO requests must be submitted at least 2 weeks in advance through the HR
portal. Requests during peak retail periods (Black Friday week, last two weeks
of December) require manager approval at least 30 days in advance. Unused PTO
may be carried over to the following year, up to a maximum of 5 days. PTO
balances exceeding the carry-over limit are forfeited on January 1. Upon
separation from the company, accrued but 

---

# Chapter 6: Embedding the Chunks

---

Convert each chunk into a vector so we can search by meaning, not keywords.

In [24]:
print(f"Embedding {len(all_chunks)} chunks with {EMBEDDING_MODEL}...")

chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = get_embeddings(chunk_texts)

print(f"\n✅ Each chunk is now a {len(chunk_embeddings[0])}-dimensional vector.")
print(f"Example (first 8 values): {chunk_embeddings[0][:8]}")

Embedding 31 chunks with text-embedding-3-small...

✅ Each chunk is now a 1536-dimensional vector.
Example (first 8 values): [-0.052398681640625, 0.0293731689453125, 0.048553466796875, 0.0028018951416015625, 0.01108551025390625, 0.0188140869140625, -0.00858306884765625, 0.01373291015625]


In [26]:
len(chunk_embeddings)

31

### Quick test: does similarity search work?

In [25]:
query = "What is the electronics return policy?"
query_embedding = get_embeddings([query])[0]

# Compare against every chunk
similarities = [(cosine_similarity(query_embedding, emb), i) for i, emb in enumerate(chunk_embeddings)]
similarities.sort(reverse=True)

print(f"Query: \"{query}\"\n")
print("Top 3 matches:")
for score, idx in similarities[:3]:
    c = all_chunks[idx]
    print(f"  {score:.4f} | {c['source']} | \"{c['text'][:100]}...\"\n")

print(f"Worst match: {similarities[-1][0]:.4f} | {all_chunks[similarities[-1][1]]['source']}")
print(f"\n💡 Finds the right content — but computing against every chunk is slow.")
print(f"   That's what vector databases solve.")

Query: "What is the electronics return policy?"

Top 3 matches:
  0.7423 | return_policy.txt | "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors, peripherals) have ..."

  0.6392 | return_policy.txt | "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
Standard items may be ..."

  0.5935 | return_policy.txt | "9. Exceptions
The store manager has discretion to approve returns outside of standard policy for cus..."

Worst match: 0.1143 | employee_handbook.txt

💡 Finds the right content — but computing against every chunk is slow.
   That's what vector databases solve.


---

# Chapter 7: Vector Database (ChromaDB)

---

Store embeddings in a database built for fast similarity search.

In [ ]:
import chromadb

chroma_client = chromadb.Client()

# Clean slate if re-running
try: chroma_client.delete_collection("techstore_docs")
except: pass

collection = chroma_client.create_collection(
    name="techstore_docs",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=chunk_embeddings,
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]} for c in all_chunks],
)

print(f"✅ {collection.count()} chunks stored in ChromaDB")

In [ ]:
def search(query, n_results=3):
    """Search ChromaDB for chunks relevant to the query."""
    query_embedding = get_embeddings([query])[0]
    return collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )


# Test with several questions
for q in ["Can I return a laptop after 20 days?",
          "How many vacation days after 4 years?",
          "Best laptop for machine learning?",
          "Remote work policy?"]:
    r = search(q, n_results=2)
    print(f"\nQ: {q}")
    for i in range(len(r['documents'][0])):
        score = 1 - r['distances'][0][i]
        src = r['metadatas'][0][i]['source']
        print(f"  [{score:.3f}] {src}: \"{r['documents'][0][i][:80]}...\"")

---

# Chapter 8: Retrieval Strategies

---

How many chunks to retrieve, and how to scope your search.

In [ ]:
# Effect of K on results
query = "What benefits does TechStore offer employees?"
for k in [1, 3, 5, 8]:
    r = search(query, n_results=k)
    sources = [m['source'] for m in r['metadatas'][0]]
    scores = [f"{1-d:.3f}" for d in r['distances'][0]]
    print(f"K={k}: {sources}")
    print(f"     {scores}\n")

In [ ]:
# Metadata filtering — search only within one document
query = "What is the return policy?"
q_emb = get_embeddings([query])[0]

r_all = collection.query(query_embeddings=[q_emb], n_results=2, include=["metadatas"])
r_hr = collection.query(query_embeddings=[q_emb], n_results=2,
                        where={"source": "employee_handbook.txt"}, include=["metadatas"])

print(f"All docs:      {[m['source'] for m in r_all['metadatas'][0]]}")
print(f"HR only:       {[m['source'] for m in r_hr['metadatas'][0]]}")
print(f"\n💡 Metadata filtering scopes search to specific docs.")

---

# Chapter 9: Building the RAG Prompt

---

Connect everything: search → assemble context → generate grounded answer.

In [ ]:
RAG_SYSTEM_PROMPT = """You are a helpful assistant that answers questions about TechStore.

RULES:
- Answer ONLY using the provided context
- If the answer is not in the context, say "I don't have that information in my knowledge base."
- Do NOT use your general knowledge — only what's in the context
- Cite the source document when possible
- Be concise and direct"""


def rag_query(question, n_chunks=3, show_context=False):
    """Complete RAG pipeline: search → assemble → generate."""
    results = search(question, n_results=n_chunks)
    
    # Assemble context
    context_parts = []
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        source = results['metadatas'][0][i]['source']
        score = 1 - results['distances'][0][i]
        context_parts.append(f"[Source: {source}, Score: {score:.3f}]\n{doc}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    user_prompt = f"""Context (retrieved from knowledge base):
---
{context}
---

Question: {question}"""
    
    if show_context:
        print("RETRIEVED CONTEXT:")
        print("═" * 60)
        for i, part in enumerate(context_parts):
            print(f"\nChunk {i+1}:")
            print(textwrap.fill(part[:250], width=80))
        print(f"\n{'═' * 60}\nANSWER:\n{'─' * 60}")
    
    return ask(user_prompt, system_prompt=RAG_SYSTEM_PROMPT, temperature=0)


print("✅ RAG pipeline ready.")

In [ ]:
# The same questions that FAILED in Chapter 1 — now they work

print("Question 1: Electronics return policy")
print("═" * 60)
rag_query("What is TechStore's return policy for electronics?", show_context=True)

In [ ]:
print("Question 2: PTO after 4 years")
print("═" * 60)
rag_query("How many PTO days does a TechStore employee with 4 years tenure get?", show_context=True)

In [ ]:
# Test across all documents
for q in [
    "Can I return a laptop I bought 20 days ago?",
    "What laptop should I buy for machine learning?",
    "Is remote work allowed for retail employees?",
    "What happens if my order arrives damaged?",
    "What is the employee discount?",
    "Are gift cards refundable?",
    "What monitor has USB-C charging?",
    "How long is parental leave for the primary caregiver?",
]:
    print(f"\n{'═' * 60}")
    print(f"Q: {q}")
    print(f"{'─' * 60}")
    rag_query(q)

In [ ]:
# Question NOT in the documents — should say "I don't know"
print("Question NOT in our docs:")
print("═" * 60)
rag_query("What is TechStore's stock price?", show_context=True)
print("\n💡 No hallucination — says it doesn't have the info.")

---

# Chapter 10: Debugging RAG

---

When RAG gives a bad answer, check: (1) Were the right chunks retrieved? (2) Did the LLM use them correctly?

Always debug retrieval first.

In [ ]:
def debug_rag(question, n_chunks=5):
    """Inspect retrieval scores before generating."""
    print(f"🔍 \"{question}\"")
    print("═" * 60)
    
    results = search(question, n_results=n_chunks)
    
    for i in range(len(results['documents'][0])):
        score = 1 - results['distances'][0][i]
        src = results['metadatas'][0][i]['source']
        tag = "🟢" if score > 0.5 else "🟡" if score > 0.35 else "🔴"
        print(f"  [{i+1}] {tag} {score:.4f} | {src}")
        print(f"      \"{results['documents'][0][i][:120]}...\"\n")
    
    top = 1 - results['distances'][0][0]
    if top > 0.5:
        print(f"✅ Top chunk relevant ({top:.3f}). Bad answer → generation problem.")
    elif top > 0.35:
        print(f"⚠️  Moderate ({top:.3f}). Try: more chunks, rephrase query.")
    else:
        print(f"❌ Low ({top:.3f}). Info probably not in documents.")
    
    print(f"\n{'═' * 60}\nANSWER:\n{'─' * 60}")
    return rag_query(question, n_chunks=min(n_chunks, 3))


debug_rag("What is the return window for electronics?")

In [ ]:
# Different wording — embedding search handles synonyms
debug_rag("Can I send back my gadget if I don't like it?")

In [ ]:
# Info that doesn't exist
debug_rag("What is TechStore's revenue?")

---

# Summary

---

### What we built

| Step | Tool | What it does |
|---|---|---|
| Read documents | Python `open()` | Raw text from files |
| Chunk | LangChain `RecursiveCharacterTextSplitter` | Split at sentence boundaries |
| Embed | OpenAI `text-embedding-3-small` | Text → 1536-dim vectors |
| Store | ChromaDB | Fast similarity search |
| Search | `collection.query()` | Find relevant chunks |
| Generate | GPT-4o-mini + grounded prompt | Answer from context only |
| Debug | `debug_rag()` | Inspect retrieval before generation |

### Standalone script

```bash
python rag_pipeline.py "What is the return policy for electronics?"
python rag_pipeline.py "How many PTO days after 4 years?" --debug
```

### Next → RAG Part 2

Advanced retrieval (hybrid search, HyDE, re-ranking), evaluation (RAGAS), production vector databases.